# DR-LiteNet — Google Colab Runner

**Project:** DR-LiteNet: A Lightweight Explainable Hybrid CNN Framework for Imbalanced Diabetic Retinopathy Grading Using Adaptive SMOTE Fusion

---

## Quick-Start Guide

| Step | Action |
|------|--------|
| 1 | Upload your APTOS 2019 dataset to Google Drive (see **Section 5** for exact folder structure) |
| 2 | Open this notebook in Google Colab (**Runtime → Change runtime type → GPU**) |
| 3 | Run **Section 1** (Drive mount) |
| 4 | Run **Section 2** (GPU check) |
| 5 | Run **Section 3** (Clone repo) |
| 6 | Run **Section 4** (Install deps) |
| 7 | Run **Section 5** (Dataset paths + symlinks) |
| 8 | Run **Section 6** (Config patch) |
| 9 | Run **Section 7** only — train the proposed DR-LiteNet model |
| 10 | After training finishes, run **Section 8** (Evaluate) |
| 11 | Run baselines **one at a time** in **Section 9** |
| 12 | Run ablations **one at a time** in **Section 10** |
| 13 | Run **Section 11** (Comparison tables) |
| 14 | Run **Section 12** (Grad-CAM / Explainability) |
| 15 | Run **Section 13** (Inference) |

---

### Important Colab Free-Tier Notes
- **Do NOT run all cells at once.** Each major experiment is in its own section.
- Colab free tier disconnects after ~12 hours. Checkpoints are saved to Drive automatically.
- If runtime disconnects mid-training, re-run Sections 1–6 (setup), then resume from checkpoint.
- Run **memory cleanup cells** between heavy experiments.
- Recommended training order per session: one model at a time.

---
## Section 1 — Mount Google Drive & Define Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
import os
from pathlib import Path

# ── Edit these if your Drive folder name differs ──────────────────────────
PROJECT_NAME    = 'DRLiteNet'
DRIVE_ROOT      = f'/content/drive/MyDrive/{PROJECT_NAME}'
# ──────────────────────────────────────────────────────────────────────────

# Subdirectories inside Drive
DATA_ROOT        = f'{DRIVE_ROOT}/datasets'          # raw APTOS / IDRiD data
CHECKPOINT_ROOT  = f'{DRIVE_ROOT}/checkpoints'       # model checkpoints
OUTPUT_ROOT      = f'{DRIVE_ROOT}/outputs'           # figures, tables, predictions
LOG_ROOT         = f'{DRIVE_ROOT}/logs'              # training CSV logs
RESULTS_ROOT     = f'{DRIVE_ROOT}/results'           # aggregated results

# Repo directory inside Colab runtime (ephemeral)
REPO_DIR = f'/content/{PROJECT_NAME}'

# Create all Drive directories
for d in [DATA_ROOT, CHECKPOINT_ROOT, OUTPUT_ROOT, LOG_ROOT, RESULTS_ROOT,
          f'{CHECKPOINT_ROOT}/main_model',
          f'{CHECKPOINT_ROOT}/baselines',
          f'{CHECKPOINT_ROOT}/ablations',
          f'{OUTPUT_ROOT}/main_model',
          f'{OUTPUT_ROOT}/baselines',
          f'{OUTPUT_ROOT}/ablations',
          f'{OUTPUT_ROOT}/inference',
          f'{OUTPUT_ROOT}/visualizations']:
    os.makedirs(d, exist_ok=True)

print('Drive paths:')
print(f'  DRIVE_ROOT      : {DRIVE_ROOT}')
print(f'  DATA_ROOT       : {DATA_ROOT}')
print(f'  CHECKPOINT_ROOT : {CHECKPOINT_ROOT}')
print(f'  OUTPUT_ROOT     : {OUTPUT_ROOT}')
print(f'  LOG_ROOT        : {LOG_ROOT}')
print(f'  REPO_DIR        : {REPO_DIR}')
print('All Drive directories ready.')

---
## Section 2 — GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu_name}  ({vram_gb:.1f} GB VRAM)')
    print(f'CUDA: {torch.version.cuda}')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU.')

print(f'PyTorch: {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

---
## Section 3 — Clone GitHub Repository

In [ ]:
import os

# ── Replace with your actual GitHub URL ───────────────────────────────────
GITHUB_URL = 'https://github.com/Username/DRLiteNet.git'
# ──────────────────────────────────────────────────────────────────────────

REPO_DIR = f'/content/{PROJECT_NAME}'

# Disable credential prompts — required in Colab (no terminal available)
!git config --global credential.helper ""
!git config --global core.askPass ""

if os.path.isdir(REPO_DIR):
    print(f'Repo already exists at {REPO_DIR} — pulling latest changes...')
    !GIT_TERMINAL_PROMPT=0 git -C {REPO_DIR} pull
else:
    print(f'Cloning {GITHUB_URL} → {REPO_DIR}...')
    !GIT_TERMINAL_PROMPT=0 git clone {GITHUB_URL} {REPO_DIR}

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

---
## Section 4 — Install Dependencies

> ⚠️ Colab already ships with PyTorch, NumPy, Pandas, Matplotlib, scikit-learn, PIL, and OpenCV.
> Only **imbalanced-learn** and **seaborn** need to be installed on top.

In [ ]:
import subprocess, sys

# ── Strategy: install only what Colab is missing ─────────────────────────
# Colab pre-installs: torch, torchvision, numpy, pandas, matplotlib,
#                     scikit-learn, opencv, Pillow, tqdm, scipy, seaborn
# Missing on Colab: imbalanced-learn
# ──────────────────────────────────────────────────────────────────────────

print('Installing missing packages...')
!pip install imbalanced-learn==0.12.3 -q

# Verify all critical imports
print('\nVerifying imports...')
import_checks = [
    ('torch',             'PyTorch'),
    ('torchvision',       'TorchVision'),
    ('cv2',               'OpenCV'),
    ('numpy',             'NumPy'),
    ('pandas',            'Pandas'),
    ('sklearn',           'scikit-learn'),
    ('imblearn',          'imbalanced-learn'),
    ('matplotlib',        'Matplotlib'),
    ('seaborn',           'Seaborn'),
    ('PIL',               'Pillow'),
    ('tqdm',              'tqdm'),
    ('scipy',             'SciPy'),
]
all_ok = True
for mod, name in import_checks:
    try:
        __import__(mod)
        print(f'  [OK] {name}')
    except ImportError:
        print(f'  [MISSING] {name}  →  run: pip install {mod}')
        all_ok = False

print('\nAll dependencies OK.' if all_ok else '\nSome dependencies missing — see above.')

---
## Section 5A — Download APTOS 2019 Dataset via Kaggle API

**Dataset:** [mariaherrerot/aptos2019](https://www.kaggle.com/datasets/mariaherrerot/aptos2019)

**One-time setup — Token file:**
1. [kaggle.com](https://www.kaggle.com) → Profile → **Settings** → **Create New Token**
2. Token string copy করুন (শুরু হয় `KGAT_` দিয়ে)
3. একটা plain text file বানান নাম দিন `kaggle_token.txt`
4. ভেতরে শুধু token string paste করুন — আর কিছু না
5. Google Drive-এ upload করুন: `MyDrive/DRLiteNet/kaggle_token.txt`

তারপর নিচের দুটো cell run করুন।

> **⚠️ সতর্কতা:** `kaggle_token.txt` কাউকে share করবেন না, GitHub-এ push করবেন না
> **⏱️ সময়:** ~5–10 মিনিট (download + copy to Drive)
> **✅ পরের session-এ:** Section 5A skip করুন — Drive-এ images থেকে যাবে

In [ ]:
# ── Step 1: Install Kaggle API & configure credentials ───────────────────
#
# New Kaggle API token format (starts with KGAT_...)
# Save your token to Google Drive as a plain text file:
#   MyDrive/DRLiteNet/kaggle_token.txt
#   (file contains just the token string, nothing else)
# ─────────────────────────────────────────────────────────────────────────

import os, shutil
from pathlib import Path

# Install kaggle package
!pip install kaggle -q

# ── Read token from Drive ─────────────────────────────────────────────────
TOKEN_FILE_ON_DRIVE = f'{DRIVE_ROOT}/kaggle_token.txt'
token_path = Path(TOKEN_FILE_ON_DRIVE)

if not token_path.exists():
    raise FileNotFoundError(
        f'\n\nkaggle_token.txt not found at {TOKEN_FILE_ON_DRIVE}\n'
        '  → Create a plain text file named kaggle_token.txt\n'
        '  → Paste your Kaggle API token (starts with KGAT_) inside\n'
        f'  → Upload to Google Drive at: {TOKEN_FILE_ON_DRIVE}'
    )

token = token_path.read_text().strip()

if not token.startswith('KGAT_'):
    raise ValueError(
        f'Token format looks wrong. Expected it to start with "KGAT_"\n'
        f'Got: {token[:10]}...\n'
        'Generate a new token from kaggle.com → Settings → Create New Token'
    )

# ── Write to ~/.kaggle/access_token (new format) ──────────────────────────
kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
access_token_file = kaggle_dir / 'access_token'
access_token_file.write_text(token)
access_token_file.chmod(0o600)

# Also set as environment variable (belt-and-suspenders)
os.environ['KAGGLE_API_TOKEN'] = token

# ── Verify ────────────────────────────────────────────────────────────────
import kaggle
print('Kaggle credentials configured successfully.')
print(f'  Token file : {TOKEN_FILE_ON_DRIVE}')
print(f'  Saved to   : {access_token_file}')
print(f'  Token      : {token[:8]}...{token[-4:]}  (partially hidden)')

In [ ]:
# ── Step 2: Download APTOS 2019 & organize into project folder structure ─
#
# Dataset  : https://www.kaggle.com/datasets/mariaherrerot/aptos2019
# Command  : kaggle datasets download -d mariaherrerot/aptos2019
#
# Expected raw layout after unzip (auto-detected below):
#   train_1.csv / train.csv   ← labeled rows (id_code, diagnosis)
#   train_images/*.png        ← fundus images
#
# We create a stratified 80 / 10 / 10 split and save to Drive:
#   DATA_ROOT/train_1.csv           (≈2929 rows)
#   DATA_ROOT/valid.csv             (≈  366 rows)
#   DATA_ROOT/test.csv              (≈  367 rows)
#   DATA_ROOT/train_images/train_images/*.png
#   DATA_ROOT/val_images/val_images/*.png
#   DATA_ROOT/test_images/test_images/*.png
# ─────────────────────────────────────────────────────────────────────────

import os, shutil
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# ── 1. Download ───────────────────────────────────────────────────────────
RAW_DIR = Path('/content/aptos_raw')
RAW_DIR.mkdir(exist_ok=True)

KAGGLE_DATASET = 'mariaherrerot/aptos2019'
ZIP_NAME       = 'aptos2019.zip'

# Skip download if already present from a previous run
already_downloaded = any(RAW_DIR.iterdir()) if RAW_DIR.exists() else False

if already_downloaded:
    print('[SKIP] Raw files already present in /content/aptos_raw — skipping download.')
else:
    print(f'Downloading {KAGGLE_DATASET} from Kaggle ...')
    !kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_DIR} --unzip --quiet
    print('Download + unzip complete.')

# ── 2. Auto-detect CSV and image folder ──────────────────────────────────
# The dataset may have files at root or inside a subfolder — handle both.
def find_file(root: Path, pattern: str):
    matches = list(root.rglob(pattern))
    return matches[0] if matches else None

raw_csv = find_file(RAW_DIR, 'train_1.csv') or find_file(RAW_DIR, 'train.csv')
raw_img_dir = find_file(RAW_DIR, 'train_images')

if raw_csv is None:
    raise FileNotFoundError(
        'Could not find train_1.csv or train.csv in the downloaded dataset.\n'
        f'Contents of {RAW_DIR}:\n' + '\n'.join(str(p) for p in RAW_DIR.rglob('*') if p.is_file())
    )
if raw_img_dir is None or not raw_img_dir.is_dir():
    raise FileNotFoundError(
        'Could not find train_images/ folder in the downloaded dataset.'
    )

n_imgs = len(list(raw_img_dir.glob('*.png')))
df_raw = pd.read_csv(raw_csv)
print(f'\nDetected CSV  : {raw_csv.relative_to(RAW_DIR)}  ({len(df_raw)} rows)')
print(f'Detected imgs : {raw_img_dir.relative_to(RAW_DIR)}/  ({n_imgs} images)')

# ── 3. Stratified 80 / 10 / 10 split ─────────────────────────────────────
df = df_raw[['id_code', 'diagnosis']].dropna()

train_df, tmp_df = train_test_split(
    df, test_size=0.20, stratify=df['diagnosis'], random_state=42
)
val_df, test_df = train_test_split(
    tmp_df, test_size=0.50, stratify=tmp_df['diagnosis'], random_state=42
)

print(f'\nSplit sizes:')
print(f'  train_1.csv : {len(train_df):5d} rows')
print(f'  valid.csv   : {len(val_df):5d} rows')
print(f'  test.csv    : {len(test_df):5d} rows')
print(f'\nClass distribution (train):')
names = {0:'No DR', 1:'Mild', 2:'Moderate', 3:'Severe', 4:'Proliferative DR'}
for c, n in train_df['diagnosis'].value_counts().sort_index().items():
    print(f'  Class {c} ({names[c]:20s}): {n:5d}')

# ── 4. Save CSVs to Drive ─────────────────────────────────────────────────
DATA_PATH = Path(DATA_ROOT)
DATA_PATH.mkdir(parents=True, exist_ok=True)

train_df.to_csv(DATA_PATH / 'train_1.csv', index=False)
val_df.to_csv  (DATA_PATH / 'valid.csv',   index=False)
test_df.to_csv (DATA_PATH / 'test.csv',    index=False)
print('\nCSVs saved to Google Drive.')

# ── 5. Copy images into the nested folder structure ───────────────────────
SPLIT_MAP = {
    'train_images/train_images': set(train_df['id_code']),
    'val_images/val_images':     set(val_df['id_code']),
    'test_images/test_images':   set(test_df['id_code']),
}

for rel_path, id_set in SPLIT_MAP.items():
    dest_dir = DATA_PATH / rel_path
    existing = len(list(dest_dir.glob('*.png'))) if dest_dir.exists() else 0
    if existing == len(id_set):
        print(f'[SKIP] {rel_path}/ already has {existing} images.')
        continue
    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f'Copying {len(id_set)} images → {rel_path}/ ...', end=' ', flush=True)
    copied = 0
    for id_code in id_set:
        src = raw_img_dir / f'{id_code}.png'
        dst = dest_dir / f'{id_code}.png'
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            copied += 1
    print(f'done ({copied} copied)')

# ── 6. Clean up temp download to free Colab disk space ───────────────────
shutil.rmtree(RAW_DIR, ignore_errors=True)
print('\nTemp download folder removed.')
print('\n✓ Dataset ready on Google Drive. Proceed to Section 5 to verify.')

---
## Section 5 — Dataset Setup & Path Verification

### Expected Google Drive structure

Upload your APTOS 2019 dataset to Drive **exactly** like this:

```
/content/drive/MyDrive/DRLiteNet/datasets/
    train_1.csv            ← 2929 rows  (id_code, diagnosis)
    valid.csv              ← 365  rows
    test.csv               ← 868  rows
    train_images/
        train_images/      ← *.png  (2930 images)
    val_images/
        val_images/        ← *.png  (366 images)
    test_images/
        test_images/       ← *.png  (366 images)

    [optional] idrid/
        images/            ← IDRiD *.jpg images
        labels.csv         ← columns: id_code, diagnosis
```

### How the symlinks work
`src/config.py` derives all paths from the repo root automatically.  
We symlink the repo's `data/`, `experiments/`, and `outputs/` directories  
to Google Drive so **all saves go directly to Drive**.

In [ ]:
import os
from pathlib import Path

REPO_DIR = f'/content/{PROJECT_NAME}'

# Map:  repo-relative path  →  Google Drive path
SYMLINK_MAP = {
    f'{REPO_DIR}/data':        DATA_ROOT,
    f'{REPO_DIR}/experiments': CHECKPOINT_ROOT,
    f'{REPO_DIR}/outputs':     OUTPUT_ROOT,
}

for link_path, target_path in SYMLINK_MAP.items():
    link = Path(link_path)
    target = Path(target_path)
    target.mkdir(parents=True, exist_ok=True)

    if link.is_symlink():
        if str(link.resolve()) == str(target):
            print(f'  [SKIP] {link.name}/ symlink already correct')
        else:
            link.unlink()
            link.symlink_to(target)
            print(f'  [UPDATED] {link.name}/ → {target}')
    elif link.exists():
        # Non-symlink dir exists (e.g. from .gitkeep) — remove and re-link
        import shutil
        shutil.rmtree(link_path, ignore_errors=True)
        link.symlink_to(target)
        print(f'  [LINKED] {link.name}/ → {target}')
    else:
        link.symlink_to(target)
        print(f'  [LINKED] {link.name}/ → {target}')

print('\nSymlinks ready. Verifying dataset paths...')
print()

In [ ]:
import os
from pathlib import Path

REQUIRED = {
    'train_1.csv':                   f'{DATA_ROOT}/train_1.csv',
    'valid.csv':                     f'{DATA_ROOT}/valid.csv',
    'test.csv':                      f'{DATA_ROOT}/test.csv',
    'train_images/train_images/':    f'{DATA_ROOT}/train_images/train_images',
    'val_images/val_images/':        f'{DATA_ROOT}/val_images/val_images',
    'test_images/test_images/':      f'{DATA_ROOT}/test_images/test_images',
}

all_present = True
for label, path in REQUIRED.items():
    p = Path(path)
    if p.exists():
        if p.is_dir():
            n = len(list(p.iterdir()))
            print(f'  [OK] {label}  ({n} files)')
        else:
            import pandas as pd
            df = pd.read_csv(p)
            print(f'  [OK] {label}  ({len(df)} rows)')
    else:
        print(f'  [MISSING] {label}')
        print(f'           Expected: {path}')
        all_present = False

print()
if all_present:
    print('All required dataset files found. Ready to train.')
else:
    print('ERROR: Some dataset files are missing.')
    print('Upload your APTOS 2019 dataset to Google Drive as described in Section 5.')

# Print class distribution
if Path(f'{DATA_ROOT}/train_1.csv').exists():
    import pandas as pd
    df = pd.read_csv(f'{DATA_ROOT}/train_1.csv')
    print('\nTraining set class distribution:')
    names = {0:'No DR', 1:'Mild', 2:'Moderate', 3:'Severe', 4:'Proliferative DR'}
    for c, n in df['diagnosis'].value_counts().sort_index().items():
        print(f'  Class {c} ({names[c]:20s}): {n:5d} images')

---
## Section 6 — Training Hyperparameters (Edit Before Training)

Edit the variables below to control training.  
These values are passed directly to `src/main.py` via command-line arguments —  
**no changes to `src/config.py` are needed.**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  EDIT THIS CELL BEFORE RUNNING ANY TRAINING
# ═══════════════════════════════════════════════════════════════════════════

# --- Image & model ---
IMAGE_SIZE    = 224          # do not change (backbone pretrained at 224)
NUM_CLASSES   = 5
NUM_FOLDS     = 5

# --- Batch size ---
# T4 (16 GB): BATCH_SIZE=16 is safe for Phase 1; 8–12 for Phase 2 fine-tuning
# P100 (16 GB): BATCH_SIZE=32 usually works
BATCH_SIZE    = 16           # ← lower to 8 if you get OOM errors

# --- Epochs ---
# Full training: P1=20, P2=30 (may need 2–3 Colab sessions)
# Quick test   : P1=5,  P2=5  (verify everything works first)
P1_EPOCHS     = 20           # Phase 1: frozen backbone + ADASYN + head training
P2_EPOCHS     = 30           # Phase 2: end-to-end fine-tuning
BASELINE_EPOCHS = 50         # for pure-CNN baselines (single-phase training)

# --- DataLoader workers ---
# Colab: 2 is usually optimal; set to 0 if you see multiprocessing errors
NUM_WORKERS   = 2

# ═══════════════════════════════════════════════════════════════════════════

import os; os.chdir(f'/content/{PROJECT_NAME}')
print('Hyperparameters set:')
print(f'  BATCH_SIZE   = {BATCH_SIZE}')
print(f'  P1_EPOCHS    = {P1_EPOCHS}  (Phase 1: frozen backbone + SMOTE)')
print(f'  P2_EPOCHS    = {P2_EPOCHS}  (Phase 2: end-to-end fine-tuning)')
print(f'  BASELINE_EPOCHS = {BASELINE_EPOCHS}')
print(f'  NUM_WORKERS  = {NUM_WORKERS}')
print()
print('Working directory:', os.getcwd())

import sys
if f'/content/{PROJECT_NAME}' not in sys.path:
    sys.path.insert(0, f'/content/{PROJECT_NAME}')

---
## Section 7 — Train Proposed Model: DR-LiteNet

Two backbone variants are available:
- **`dr_litenet_effb0`** — EfficientNetB0 (4.69M params) — **recommended, default**
- **`dr_litenet_mobv3`** — MobileNetV3-Small (1.25M params) — faster, lower accuracy

Training is **two-phase**:
1. **Phase 1** — backbone frozen → extract features → ADASYN SMOTE → train dense head
2. **Phase 2** — unfreeze backbone → end-to-end fine-tuning

Checkpoints are saved automatically to:  
`/content/drive/MyDrive/DRLiteNet/checkpoints/<model_name>/fold_<k>/checkpoints/best_model.pth`

> ⚠️ Run **one fold at a time** if you are worried about session timeout.  
> Use `--fold 0` through `--fold 4` to train individual folds.

In [ ]:
# ── DR-LiteNet (EfficientNetB0) — ALL 5 FOLDS ───────────────────────────
# Comment out this cell and use the 'single fold' cell below if you want
# to train fold-by-fold across multiple sessions.

import os; os.chdir(f'/content/{PROJECT_NAME}')

!python src/main.py \
    --mode train \
    --model dr_litenet_effb0 \
    --p1_epochs {P1_EPOCHS} \
    --p2_epochs {P2_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

In [ ]:
# ── DR-LiteNet (EfficientNetB0) — SINGLE FOLD (use when nearing timeout) ─
# Change FOLD_IDX to 0, 1, 2, 3, or 4 to train one fold at a time.

import os; os.chdir(f'/content/{PROJECT_NAME}')

FOLD_IDX = 0   # ← change to the fold you want to train

!python src/main.py \
    --mode train \
    --model dr_litenet_effb0 \
    --fold {FOLD_IDX} \
    --p1_epochs {P1_EPOCHS} \
    --p2_epochs {P2_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

In [ ]:
# ── DR-LiteNet (MobileNetV3-Small) — ALL 5 FOLDS ────────────────────────
# Run this AFTER training dr_litenet_effb0 (separate session recommended)

import os; os.chdir(f'/content/{PROJECT_NAME}')

!python src/main.py \
    --mode train \
    --model dr_litenet_mobv3 \
    --p1_epochs {P1_EPOCHS} \
    --p2_epochs {P2_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

In [ ]:
# ── Memory cleanup after training ────────────────────────────────────────
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU memory freed. Available: {torch.cuda.memory_reserved(0)/1e9:.1f} GB reserved')
print('Memory cleanup done.')

---
## Section 8 — Evaluate Proposed Model

Loads best checkpoints from all trained folds, computes all metrics, and generates:
- Confusion matrices (per fold + averaged)
- ROC curves with AUC
- Training/validation loss and QWK curves
- Class distribution before/after SMOTE
- Per-fold CSV + averaged CSV tables

All files saved to:  
`/content/drive/MyDrive/DRLiteNet/outputs/figures/`  
`/content/drive/MyDrive/DRLiteNet/outputs/tables/`

In [ ]:
# ── Evaluate DR-LiteNet (EfficientNetB0) ─────────────────────────────────
import os; os.chdir(f'/content/{PROJECT_NAME}')

!python src/main.py \
    --mode evaluate \
    --model dr_litenet_effb0 \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

print('\nResults saved to Google Drive:')
print(f'  Figures : {OUTPUT_ROOT}/figures/')
print(f'  Tables  : {OUTPUT_ROOT}/tables/')

In [ ]:
# ── Evaluate DR-LiteNet (MobileNetV3-Small) ──────────────────────────────
import os; os.chdir(f'/content/{PROJECT_NAME}')

!python src/main.py \
    --mode evaluate \
    --model dr_litenet_mobv3 \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

In [ ]:
# ── Print averaged results from CSV ──────────────────────────────────────
import pandas as pd
from pathlib import Path

avg_csv = Path(OUTPUT_ROOT) / 'tables' / 'averaged_results.csv'
if avg_csv.exists():
    df = pd.read_csv(avg_csv)
    print('DR-LiteNet EfficientNetB0 — averaged results across folds:')
    print(df.to_string(index=False))
else:
    print(f'No averaged_results.csv found at {avg_csv}')
    print('Run the evaluate cell above first.')

---
## Section 9 — Baseline Model Training

Run each baseline in its own cell, **one per session** if possible.
All baselines use identical 5-fold splits (seed=42) and identical preprocessing.

| Baseline | Script | What it tests |
|---|---|---|
| `effb0_baseline` | `baselines/train_effb0_baseline.py` | EfficientNetB0, deep features only, no SMOTE |
| `effb0_weighted` | `baselines/train_effb0_weighted.py` | EfficientNetB0 + class-weighted CE loss |
| `mobv3_baseline` | `baselines/train_mobv3_baseline.py` | MobileNetV3-Small, deep features only |
| `resnet50_baseline` | `baselines/train_resnet50_baseline.py` | ResNet50 accuracy ceiling reference |

> ⚠️ Run one baseline per Colab session to avoid timeout. Re-run Sections 1–6 to reconnect.

In [ ]:
# ── Baseline 1: EfficientNetB0 (no handcrafted features, no SMOTE) ───────
import os; os.chdir(f'/content/{PROJECT_NAME}')

!python baselines/train_effb0_baseline.py \
    --all_folds \
    --epochs {BASELINE_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

# Cleanup
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print('Baseline 1 done. Checkpoints saved to Drive.')

In [ ]:
# ── Baseline 2: EfficientNetB0 + class-weighted CE loss (no SMOTE) ───────
import os; os.chdir(f'/content/{PROJECT_NAME}')

!python baselines/train_effb0_weighted.py \
    --all_folds \
    --epochs {BASELINE_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

import gc, torch; gc.collect(); torch.cuda.empty_cache()
print('Baseline 2 done.')

In [ ]:
# ── Baseline 3: MobileNetV3-Small (no handcrafted features, no SMOTE) ────
import os; os.chdir(f'/content/{PROJECT_NAME}')

!python baselines/train_mobv3_baseline.py \
    --all_folds \
    --epochs {BASELINE_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

import gc, torch; gc.collect(); torch.cuda.empty_cache()
print('Baseline 3 done.')

In [ ]:
# ── Baseline 4: ResNet50 (accuracy ceiling reference, no SMOTE) ──────────
# NOTE: ResNet50 has 23.5M params — use smaller batch size to avoid OOM
import os; os.chdir(f'/content/{PROJECT_NAME}')

RESNET_BATCH = min(BATCH_SIZE, 16)   # ResNet50 needs more VRAM per sample

!python baselines/train_resnet50_baseline.py \
    --all_folds \
    --epochs {BASELINE_EPOCHS} \
    --batch_size {RESNET_BATCH} \
    --num_workers {NUM_WORKERS}

import gc, torch; gc.collect(); torch.cuda.empty_cache()
print('Baseline 4 (ResNet50) done.')

---
## Section 10 — Ablation Study Training

These ablations test individual components of DR-LiteNet:

| Ablation | Script | What is removed |
|---|---|---|
| `dr_litenet_no_smote` | `baselines/train_dr_litenet_no_smote.py` | Adaptive SMOTE (tests SMOTE's contribution to RQ2) |
| `dr_litenet_weighted` | `baselines/train_dr_litenet_weighted.py` | SMOTE replaced by class-weighted CE loss |

> Note: The comparison of `dr_litenet_effb0` (full) vs. `effb0_baseline` (no handcrafted features) already tests the handcrafted branch contribution (RQ1).  
> The comparison of `dr_litenet_effb0` vs. `dr_litenet_mobv3` tests backbone choice (RQ3).

In [ ]:
# ── Ablation 1: DR-LiteNet WITHOUT Adaptive SMOTE ────────────────────────
# (Full hybrid fusion + handcrafted features, but standard CE loss, no oversampling)
import os; os.chdir(f'/content/{PROJECT_NAME}')

!python baselines/train_dr_litenet_no_smote.py \
    --all_folds \
    --epochs {BASELINE_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

import gc, torch; gc.collect(); torch.cuda.empty_cache()
print('Ablation 1 (no SMOTE) done.')

In [ ]:
# ── Ablation 2: DR-LiteNet WITH class-weighted CE loss (no SMOTE) ────────
# (Tests SMOTE vs. class-weighting for minority class sensitivity)
import os; os.chdir(f'/content/{PROJECT_NAME}')

!python baselines/train_dr_litenet_weighted.py \
    --all_folds \
    --epochs {BASELINE_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}

import gc, torch; gc.collect(); torch.cuda.empty_cache()
print('Ablation 2 (class-weighted) done.')

---
## Section 11 — Comparison Tables & Ablation Summary

Loads all `results.json` files from every trained model and fold,  
aggregates mean ± std across folds, and generates:
- `baseline_comparison.csv`
- `ablation_study.csv`
- `efficiency_table.csv` (param count + inference time)
- `sensitivity_comparison.png` (grouped bar chart)

In [ ]:
import os; os.chdir(f'/content/{PROJECT_NAME}')

!python src/main.py --mode compare

# Print the comparison table
import pandas as pd
from pathlib import Path

cmp_csv = Path(OUTPUT_ROOT) / 'tables' / 'baseline_comparison.csv'
if cmp_csv.exists():
    df = pd.read_csv(cmp_csv)
    print('\nBaseline Comparison (mean ± std across folds):')
    cols = ['model', 'accuracy', 'qwk', 'macro_f1']
    print(df[cols].to_string(index=False))
    print(f'\nFull table saved to: {cmp_csv}')
else:
    print('No comparison CSV found — run training + evaluation for at least 2 models first.')

abl_csv = Path(OUTPUT_ROOT) / 'tables' / 'ablation_study.csv'
if abl_csv.exists():
    df_a = pd.read_csv(abl_csv)
    print('\nAblation Study:')
    print(df_a[['model','accuracy','qwk','macro_f1']].to_string(index=False))

---
## Section 12 — Explainability: Grad-CAM Visualisations

Generates gradient-weighted class activation maps (Grad-CAM) on the last  
convolutional layer of the CNN branch.

Output:
- Per-class correct-prediction heatmaps (3 per class × 5 classes)
- Grid figure: `gradcam_grid_all_classes.png`
- Misclassification analysis: `misclassified/` subfolder

All saved to:  
`/content/drive/MyDrive/DRLiteNet/outputs/figures/grad_cam/`

In [ ]:
# ── Grad-CAM for DR-LiteNet (EfficientNetB0), fold 0 ─────────────────────
import os; os.chdir(f'/content/{PROJECT_NAME}')

EXPLAIN_FOLD = 0   # ← change to whichever fold you want to visualise

!python src/main.py \
    --mode explain \
    --model dr_litenet_effb0 \
    --fold {EXPLAIN_FOLD} \
    --batch_size 4 \
    --num_workers {NUM_WORKERS}

print(f'\nGrad-CAM figures saved to: {OUTPUT_ROOT}/figures/grad_cam/')

In [ ]:
# ── Display the grid figure inline ───────────────────────────────────────
from pathlib import Path
from IPython.display import Image as IPyImage, display

grid_path = Path(OUTPUT_ROOT) / 'figures' / 'grad_cam' / 'gradcam_grid_all_classes.png'

if grid_path.exists():
    print('Grad-CAM Grid (all 5 DR severity classes):')
    display(IPyImage(str(grid_path), width=900))
else:
    print(f'Grid figure not found at {grid_path}')
    print('Run the Grad-CAM cell above first.')

---
## Section 13 — Inference on New Images

Run inference on:
- A **single image file**, or
- A **directory** of fundus images (JPEG/PNG)

Output: `inference_results.json` + `inference_results.csv` with per-image predictions.

Each prediction follows the format:
```json
{"image": "patient_001.jpg",
 "predicted_class": 2,
 "predicted_label": "Moderate",
 "probabilities": [0.05, 0.10, 0.60, 0.15, 0.10],
 "grad_cam_path": "outputs/grad_cam/inference/patient_001_gradcam.png"}
```

In [ ]:
# ── Inference on the validation set ──────────────────────────────────────
# Change INPUT_PATH to any folder or single image file
import os; os.chdir(f'/content/{PROJECT_NAME}')

# Validation images are already available through the Drive symlink
INPUT_PATH = f'{DATA_ROOT}/val_images/val_images'

!python src/main.py \
    --mode infer \
    --model dr_litenet_effb0 \
    --input "{INPUT_PATH}" \
    --output "{OUTPUT_ROOT}/inference" \
    --gradcam

print(f'\nPredictions saved to: {OUTPUT_ROOT}/inference/')

In [ ]:
# ── IDRiD External Validation (skip if IDRiD not uploaded) ───────────────
import os
from pathlib import Path

idrid_csv = Path(DATA_ROOT) / 'idrid' / 'labels.csv'
if idrid_csv.exists():
    os.chdir(f'/content/{PROJECT_NAME}')
    !python src/main.py \
        --mode infer \
        --model dr_litenet_effb0 \
        --idrid
    print(f'External validation results saved to: {OUTPUT_ROOT}/tables/external_validation.csv')
else:
    print('[SKIP] IDRiD dataset not found.')
    print(f'To enable: upload IDRiD data to {DATA_ROOT}/idrid/')
    print('  Required files: idrid/images/*.jpg  and  idrid/labels.csv (columns: id_code, diagnosis)')

In [ ]:
# ── Preview inference results ─────────────────────────────────────────────
import pandas as pd
from pathlib import Path

csv_path = Path(OUTPUT_ROOT) / 'inference' / 'inference_results.csv'
if not csv_path.exists():
    csv_path = Path(OUTPUT_ROOT) / 'predictions' / 'inference_results.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f'Inference results ({len(df)} images):')
    print(df[['image','predicted_class','predicted_label','prob_0','prob_1','prob_2','prob_3','prob_4']].head(10).to_string(index=False))
    print(f'\nClass distribution of predictions:')
    vc = df['predicted_label'].value_counts()
    for cls, cnt in vc.items():
        print(f'  {cls:20s}: {cnt:4d}')
else:
    print('No inference CSV found. Run the inference cell above first.')

---
## Section 14 — Memory Cleanup & Session Management

Run this cell **between** heavy experiments to free GPU memory.

In [ ]:
import gc, torch

# Clear Python object cache
gc.collect()

# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    alloc  = torch.cuda.memory_allocated(0)  / 1e9
    reserv = torch.cuda.memory_reserved(0)   / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU memory — allocated: {alloc:.2f} GB | reserved: {reserv:.2f} GB | total: {total:.1f} GB')

print('Memory cleanup complete.')
print()
print('TIP: If starting a new heavy experiment (e.g. ResNet50 after EfficientNetB0),')
print('     go to Runtime → Restart runtime, then re-run Sections 1–6 before training.')

In [ ]:
# ── List all saved checkpoints on Drive ───────────────────────────────────
from pathlib import Path

print('Saved checkpoints on Google Drive:')
ckpt_root = Path(CHECKPOINT_ROOT)
pth_files = sorted(ckpt_root.rglob('*.pth'))
if pth_files:
    for p in pth_files:
        size_mb = p.stat().st_size / 1e6
        rel = p.relative_to(ckpt_root)
        print(f'  {str(rel):60s}  {size_mb:.1f} MB')
else:
    print('  No .pth checkpoints found yet. Train a model first.')

print()
print('Saved output files:')
out_root = Path(OUTPUT_ROOT)
for ext in ['*.csv', '*.json', '*.png']:
    files = list(out_root.rglob(ext))
    print(f'  {ext:8s}: {len(files):3d} files')

---
## Section 15 — Troubleshooting & FAQ

Run the cell below to print a diagnostic report.

In [ ]:
import os, sys, torch
from pathlib import Path

print('='*60)
print('DR-LiteNet Colab Diagnostic Report')
print('='*60)

# GPU
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('GPU   : NOT AVAILABLE — enable GPU in Runtime settings')

# Python path
repo = f'/content/{PROJECT_NAME}'
print(f'Repo  : {"EXISTS" if Path(repo).exists() else "MISSING — run Section 3"}')
print(f'In sys.path: {repo in sys.path}')

# Symlinks
for name, link in [('data/', f'{repo}/data'), ('experiments/', f'{repo}/experiments'),
                    ('outputs/', f'{repo}/outputs')]:
    p = Path(link)
    if p.is_symlink():
        print(f'Symlink {name:15s}: OK → {p.resolve()}')
    elif p.exists():
        print(f'Symlink {name:15s}: NOT a symlink (local dir — Drive saves NOT active)')
    else:
        print(f'Symlink {name:15s}: MISSING — run Section 5')

# Dataset
for label, path in [
    ('train_1.csv',  f'{DATA_ROOT}/train_1.csv'),
    ('train_images', f'{DATA_ROOT}/train_images/train_images'),
    ('valid.csv',    f'{DATA_ROOT}/valid.csv'),
]:
    p = Path(path)
    status = 'OK' if p.exists() else 'MISSING'
    print(f'Dataset {label:20s}: {status}')

# Common errors and fixes
print()
print('Common issues:')
print('  OOM during Phase 2    → lower BATCH_SIZE to 8 in Section 6')
print('  "Module not found"    → re-run Section 3 + set sys.path in Section 6')
print('  "Checkpoint not found" → train the model first (Section 7)')
print('  SMOTE crash            → fixed in code (auto-skips if minority class has 1 sample)')
print('  Drive not mounted      → re-run Section 1')